## Read Diseases

In [3]:
import pandas as pd
disease = pd.read_csv('../Datasets/disease/icd10_2019 - Copy.csv')

In [4]:
disease.head()

,Unnamed: 0,url,chapter,domain,sub-code,definition
0,1.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A00,Cholera
1,2.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A00.0,"Cholera due to Vibrio cholerae 01, biovar chol..."
2,3.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor"
3,4.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A00.9,"Cholera, unspecified"
4,5.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A01,Typhoid and paratyphoid fevers


## Clean Diseases

In [5]:
disease = disease[disease['definition'].duplicated() == False]

In [6]:
disease.shape

(6, 6)

In [7]:
disease.dropna(subset=['definition'], inplace=True)

In [8]:
disease = disease[disease['sub-code'].str.contains(r'\.')]

In [9]:
disease = disease[~disease["definition"].str.contains("Other|Unspecified|not elsewhere", case=False)]

In [10]:
len(disease['definition'].unique())

2

In [11]:
disease[disease['definition'].duplicated() == True]['definition'].sort_values().values

array([], dtype=object)

In [12]:
disease.head()

,Unnamed: 0,url,chapter,domain,sub-code,definition
1,2.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A00.0,"Cholera due to Vibrio cholerae 01, biovar chol..."
2,3.0,https://icd.who.int/browse10/2019/en#/A00-A09,Chapter I\r\nCertain infectious and parasitic ...,Intestinal infectious diseases\r\n(A00-A09),A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor"


In [13]:
disease.shape

(2, 6)

In [14]:
import re
disease_dic = []
for index, i in disease.iterrows():
    d = {
    "Chapter": re.search(r"\n(.*)\r", i['chapter']).group(1) if re.search(r"\n(.*)\r", i['chapter']) else None,
    "Domain": re.sub(r'[\r\n]+|\s*\(.*?\)', '', i['domain']).strip(),
    "Disease": i['definition'],
    "Sub-code": i['sub-code'],
    }
    disease_dic.append(d)

In [15]:
print(disease_dic[1]['Chapter'])

Certain infectious and parasitic diseases


In [16]:
print(disease_dic[1]['Domain'])

Intestinal infectious diseases


## Scrape Data from 🇩🇪 Federal Ministry of Health

In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()
for disease in disease_dic:
    driver.get("https://gesund.bund.de/en/icd-code-suche")
    disease["Result Link"] = []
    search = driver.find_element(By.CLASS_NAME, 'input--text')
    button = driver.find_element(By.XPATH, '//*[@id="app"]/header/a-header/div/div[1]/div/div[3]/form/a-auto-suggest/button[2]')
    search.send_keys(f"{disease['Sub-code']} {disease['Disease']}")
    button.click()
    time.sleep(2)
    res = driver.find_elements(By.CLASS_NAME, 'm-search-result__item-link')
    disease["Result Link"].append([r.get_attribute("href") for r in res][0])
driver.quit()
    

In [18]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()

for disease in disease_dic:
    disease["Article"] = []
    for link in disease["Result Link"]:
        driver.get(link)
        time.sleep(2)
        article = driver.find_element(By.XPATH, '//*[@id="standard-textseite-headline-h1"]/section[1]/div[1]')
        disease["Article"].append({
        "source": "Federal Ministry of Health",
        "content": article.text
    })
driver.quit()

In [19]:
import json

with open("diseases.json", "w", encoding="utf-8") as f:
    json.dump(disease_dic, f, ensure_ascii=False, indent=4)

## Scrape Data from Medlineplus

In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()

for disease in disease_dic:
    driver.get("https://medlineplus.gov/all_healthtopics.html")
    search = driver.find_element(By.ID, 'searchtext_primary')
    button = driver.find_element(By.XPATH, '/html/body/div[1]/header/div/div[3]/div[2]/form/div/div[2]/button')
    search.send_keys(disease['Disease'])
    button.click()
    time.sleep(2)
    res = driver.find_elements(By.CLASS_NAME, 'title')
    for r in res:
        if r.get_attribute("href") and "medlineplus.gov" in r.get_attribute("href"):
            disease["Result Link"].append(r.get_attribute("href"))
            print(r.get_attribute("href"))
driver.quit()

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()

for disease in disease_dic:
    for link in disease["Result Link"]:
      if link and "medlineplus.gov" in link:
        driver.get(link)
        time.sleep(2)
        article = driver.find_element(By.ID, 'mplus-content')
        disease["Article"].append({
    "source": "medlineplus",
    "content": article.text
})
driver.quit()

In [22]:
import json

with open("diseases.json", "w", encoding="utf-8") as f:
    json.dump(disease_dic, f, ensure_ascii=False, indent=4)

## Scrape Data from CDC

In [23]:
driver = webdriver.Chrome()
for disease in disease_dic:        
        driver.get("https://search.cdc.gov/search/")
        time.sleep(2)
        search = driver.find_element(By.XPATH, '/html/body/main/div/div/div/div/div[1]/div[1]/div[1]/div[1]/div/input[1]')
        button = driver.find_element(By.CLASS_NAME, 'cdc-fa-magnifying-glass')
        search.send_keys(disease['Disease'])
        button.click()
        time.sleep(2)
        res = driver.find_elements(By.CLASS_NAME, 'result')
        for el in res:
            a_tag = el.find_element(By.TAG_NAME, "a")
            if a_tag.get_attribute("href") and "https://wwwnc.cdc.gov/eid/article/" in a_tag.get_attribute("href"):
                link = a_tag.get_attribute("href")
                disease["Result Link"].append(link)
driver.quit()

In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()

for disease in disease_dic:
    for link in disease["Result Link"]:
      if link and "https://wwwnc.cdc.gov/eid/article/" in link:
        driver.get(link)
        time.sleep(2)
        try:
          article = driver.find_element(By.ID, 'mainbody')
          disease["Article"].append({
          "source": "CDC",
          "content": article.text
          })
          print(article.text)
        except:
            continue
driver.quit()

To the Editor: Vibrio cholerae O1, the causative agent of cholera, has 2 biotypes (classical and El Tor), which have traditionally been distinguished by phenotypic tests and by genetic differences in the major toxin-coregulated pilus (TCP) gene, the tcpA allele of the TCP cluster (1), the rstR region (regulatory region for phage lysogeny) of CTX phages (2), the type of cholera toxin (CT) produced, and the infection pattern of the disease they cause. However, 3 variants of the El Tor biotype have been described recently: Matlab (a place in Bangladesh) variants in 2002 (3), which could not be biotyped because they have a mixture of both classical and El Tor (4), Mozambique variant in 2004–2005, which has a typical El Tor genome but a tandem repeat of the classical CTX prophage in the small chromosome (5), and the altered El Tor type (a typical El Tor biotype and an El Tor CTX prophage that produces CT of the classical type) predominant in Bangladesh since 2001 (6). Hybrid vibrios have al

## Save Scraped Data

In [25]:
import json

with open("diseases.json", "w", encoding="utf-8") as f:
    json.dump(disease_dic, f, ensure_ascii=False, indent=4)

## Load Data

In [26]:
import pandas as pd
data = pd.read_json("diseases.json")

## Explore Data

In [27]:
data.head()

,Chapter,Domain,Disease,Sub-code,Result Link,Article
0,Certain infectious and parasitic diseases,Intestinal infectious diseases,"Cholera due to Vibrio cholerae 01, biovar chol...",A00.0,[https://gesund.bund.de/en/icd-code-suche/a00-...,"[{'source': 'Federal Ministry of Health', 'con..."
1,Certain infectious and parasitic diseases,Intestinal infectious diseases,"Cholera due to Vibrio cholerae 01, biovar eltor",A00.1,[https://gesund.bund.de/en/icd-code-suche/a00-...,"[{'source': 'Federal Ministry of Health', 'con..."


In [28]:
word = ""
for articles in data["Article"]:
    for article in articles:
            word = word + "" + article['content'].lower()


In [29]:
word

'you have cholera.\ncholera is caused by bacteria. its symptoms include severe runny diarrhea. vomiting is another symptom. the vomit can be watery, but also bloody. the diarrhea and vomiting results in a great deal of fluid and important nutrients being lost. this can make you very sick.\ncholera can be transmitted through contaminated water and food.to the editor: vibrio cholerae o1, the causative agent of cholera, has 2 biotypes (classical and el tor), which have traditionally been distinguished by phenotypic tests and by genetic differences in the major toxin-coregulated pilus (tcp) gene, the tcpa allele of the tcp cluster (1), the rstr region (regulatory region for phage lysogeny) of ctx phages (2), the type of cholera toxin (ct) produced, and the infection pattern of the disease they cause. however, 3 variants of the el tor biotype have been described recently: matlab (a place in bangladesh) variants in 2002 (3), which could not be biotyped because they have a mixture of both cla

In [30]:
word = word.split()

In [31]:
word

['you',
 'have',
 'cholera.',
 'cholera',
 'is',
 'caused',
 'by',
 'bacteria.',
 'its',
 'symptoms',
 'include',
 'severe',
 'runny',
 'diarrhea.',
 'vomiting',
 'is',
 'another',
 'symptom.',
 'the',
 'vomit',
 'can',
 'be',
 'watery,',
 'but',
 'also',
 'bloody.',
 'the',
 'diarrhea',
 'and',
 'vomiting',
 'results',
 'in',
 'a',
 'great',
 'deal',
 'of',
 'fluid',
 'and',
 'important',
 'nutrients',
 'being',
 'lost.',
 'this',
 'can',
 'make',
 'you',
 'very',
 'sick.',
 'cholera',
 'can',
 'be',
 'transmitted',
 'through',
 'contaminated',
 'water',
 'and',
 'food.to',
 'the',
 'editor:',
 'vibrio',
 'cholerae',
 'o1,',
 'the',
 'causative',
 'agent',
 'of',
 'cholera,',
 'has',
 '2',
 'biotypes',
 '(classical',
 'and',
 'el',
 'tor),',
 'which',
 'have',
 'traditionally',
 'been',
 'distinguished',
 'by',
 'phenotypic',
 'tests',
 'and',
 'by',
 'genetic',
 'differences',
 'in',
 'the',
 'major',
 'toxin-coregulated',
 'pilus',
 '(tcp)',
 'gene,',
 'the',
 'tcpa',
 'allele',
 'o

In [32]:
len(word)

15453

In [33]:
word_set = set(word)

In [34]:
len(word_set)

2288

In [35]:
from collections import Counter
word_counts = Counter(word)

print(word_counts.most_common(50))

[('the', 879), ('of', 642), ('and', 539), ('in', 433), ('to', 245), ('were', 227), ('strains', 226), ('cholerae', 220), ('v.', 217), ('a', 194), ('o139', 168), ('with', 166), ('by', 151), ('from', 131), ('el', 127), ('was', 125), ('cholera', 124), ('for', 122), ('tor', 114), ('o1', 100), ('that', 91), ('this', 90), ('isolated', 81), ('as', 77), ('have', 73), ('classical', 63), ('during', 62), ('which', 61), ('at', 61), ('bangladesh', 58), ('has', 56), ('ctxb', 56), ('type', 52), ('is', 51), ('gene', 50), ('all', 50), ('epidemic', 50), ('been', 47), ('recent', 47), ('these', 43), ('ctx', 42), ('genetic', 41), ('cases', 40), ('are', 40), ('nags', 40), ('had', 39), ('using', 38), ('patients', 38), ('genes', 37), ('associated', 37)]


## Clean Articles

In [36]:
import re

def clean_text(text, source):


    if source == "CDC":
        text = re.sub(r"ISSN: 1080-6059\n(.*?)\n(.*?)\n", "", text, flags=re.DOTALL)
        text = re.sub(r"\nAcknowledgment\n.*", "", text, flags=re.DOTALL)
        text = re.sub(r"This Article$", "", text)
        text = re.sub(r"(\(\d+\))", "", text, flags=re.IGNORECASE)

    elif source == "MedlinePlus":
        text = re.sub(r"You Are Here:\nHome → (.*?) →(.*?)\n", "", text, flags=re.DOTALL)
        text = re.sub(r"\nLearn More\n.*", "", text, flags=re.DOTALL)

    text = re.sub(r"\nTop\n", "", text, flags=re.IGNORECASE)
    text = re.sub(r"(Figure|Fig\.?)\s*\d+[^\n]*", "", text)
    text = re.sub(r"(Table|Tab\.?)\s*\d+[^\n]*", "", text)


    text = re.sub(r"(Acknowledgment|References|External Links).*", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"(on this page|cite this article|article metrics|downloads|table downloads)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"to the editor:?", "", text, flags=re.IGNORECASE)

    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip(".")
    text = text.strip()
    text = re.sub(r"\n+", "\n", text)
    text = text.lower()


    noise_words = [
        "pubmed", "google scholar", "external link",
        "related links", "advanced article search",
        "comments", "send to", "editors"
    ]

    for word in noise_words:
        text = re.sub(rf"\b{word}\b", "", text)


    text = re.sub(r"[^a-z0-9.,!?():\-\s/%+°]", " ", text)


    text = re.sub(r"\s+", " ", text)
    text = "Source: " + source + " Article: " + text
    return text.strip()

In [37]:
for index, row in data.iterrows():
    cleaned_articles = []
    for article in row['Article']:
        cleaned_article = clean_text(article["content"], article['source'])
        cleaned_articles.append(cleaned_article)
    
    data.at[index, 'Article'] = cleaned_articles

In [38]:
import json

with open("diseases.json", "w", encoding="utf-8") as f:
    json.dump(data.to_dict(orient="records"), f, ensure_ascii=False, indent=4)

## Chunking and metadata

In [104]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=0.5
    )

In [105]:
def chunk_text(text, splitter, overlap_size=1):
    chunks = splitter.split_text(text)
    new_chunks = []

    for i in range(len(chunks)):
        chunk = chunks[i]

        if i > 0:
            prev_part = chunks[i-1].split()[-overlap_size:]
            chunk = " ".join(prev_part) + " " + chunk

        new_chunks.append(chunk)

    return new_chunks    

In [106]:
all_chunks = []

for row in data.itertuples():
    disease = row.Disease
    chapter = row.Chapter
    domain = row.Domain

    for article in row.Article:
        chunks = chunk_text(article, splitter, overlap_size=10)
        for c in chunks:
            all_chunks.append({
                "text": c.strip(),
                "metadata": {
                "disease": disease,
                "chapter": chapter,
                "domain": domain,
                }
            })

In [107]:
all_chunks

[{'text': 'Source: Federal Ministry of Health Article: you have cholera.',
  'metadata': {'disease': 'Cholera due to Vibrio cholerae 01, biovar cholerae',
   'chapter': 'Certain infectious and parasitic diseases',
   'domain': 'Intestinal infectious diseases'}},
 {'text': 'Source: Federal Ministry of Health Article: you have cholera. cholera is caused by bacteria.',
  'metadata': {'disease': 'Cholera due to Vibrio cholerae 01, biovar cholerae',
   'chapter': 'Certain infectious and parasitic diseases',
   'domain': 'Intestinal infectious diseases'}},
 {'text': 'cholera is caused by bacteria. its symptoms include severe runny diarrhea.',
  'metadata': {'disease': 'Cholera due to Vibrio cholerae 01, biovar cholerae',
   'chapter': 'Certain infectious and parasitic diseases',
   'domain': 'Intestinal infectious diseases'}},
 {'text': 'its symptoms include severe runny diarrhea. vomiting is another symptom.',
  'metadata': {'disease': 'Cholera due to Vibrio cholerae 01, biovar cholerae',
 

## Embeeding

In [112]:
from sentence_transformers import SentenceTransformer
import faiss

def create_embedding_index(all_chunks, model):
    
    texts = [item["text"] for item in all_chunks]

    embeddings = model.encode(texts)
    faiss.normalize_L2(embeddings)
    return embeddings, texts


## Build Index

In [113]:
import numpy as np
def create_faiss_index(embeddings):
    embeddings = np.array(embeddings).astype('float32')
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(np.array(embeddings))
    return index

## Search Through Index

In [114]:
def search(query, index, model, k=1):
    query_vec = model.encode([query])
    faiss.normalize_L2(query_vec)
    scores, ids = index.search(np.array(query_vec), k)
    return scores, ids

## Using Rag Functions

In [115]:
import numpy as np
query = "What is Cholera due to Vibrio cholerae 01, biovar cholerae?"

model = "all-MiniLM-L6-v2"
model = SentenceTransformer(model).to("cuda")

embeddings, texts = create_embedding_index(all_chunks, model)
index = create_faiss_index(embeddings)
scores, ids = search(query, index, model, k=3)

for idx in ids[0]:
    print(all_chunks[idx]["text"])
    print(all_chunks[idx]["metadata"])

c:\Users\User\anaconda3\envs\ml_env\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


cholerae o1 et and 4 v. cholerae o139).
{'disease': 'Cholera due to Vibrio cholerae 01, biovar eltor', 'chapter': 'Certain infectious and parasitic diseases', 'domain': 'Intestinal infectious diseases'}
cholerae o1 et and 4 v. cholerae o139).
{'disease': 'Cholera due to Vibrio cholerae 01, biovar cholerae', 'chapter': 'Certain infectious and parasitic diseases', 'domain': 'Intestinal infectious diseases'}
v. cholerae strains a total of 63 v.
{'disease': 'Cholera due to Vibrio cholerae 01, biovar cholerae', 'chapter': 'Certain infectious and parasitic diseases', 'domain': 'Intestinal infectious diseases'}


## Record a Question

In [116]:
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np

def record_audio(filename="input.wav", duration=5, fs=16000, channels=1):
    print("Speak now...")

    audio = sd.rec(int(duration * fs), samplerate=fs, channels=channels)
    sd.wait()

    audio = np.int16(audio * 32767)

    write(filename, fs, audio)

## Cleaning Voice

In [50]:
import librosa
import numpy as np

def clean_audio(file):
    y, sr = librosa.load(file, sr=16000)
    y = librosa.effects.preemphasis(y)
    return y

## Converting Question Record into Text

In [51]:
import whisper
whisper_model = whisper.load_model("medium")

def speech_to_text(filename="input.wav", model=whisper_model):
    file = clean_audio(filename)
    time.sleep(2)
    result = model.transcribe(file, fp16=False, language="en", temperature=0, initial_prompt="Medical assistant conversation")
    return result["text"]

## Ask ollama 

In [68]:
import requests

def ask_ollama(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
           "model": "llama3:8b",
           "prompt": prompt,
           "stream": False,
           "temperature": 0.0
        }
    )
    
    return response.json()['response']

## Converting Answer Into Speech with Piper

In [53]:
import re

def humanize_text(text):
    
    text = re.sub(r"[*|#|-|_]", "", text)

    text = text.replace("\n", ". ")

    text = text.replace(",", ", ... ")
    
    text = text.replace(".", ". ... ")

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [54]:
import os
import subprocess

PIPER_PATH = r"C:\Users\User\AppData\Local\Programs\piper\piper.exe"
MODEL_PATH = r"C:\Users\User\AppData\Local\Programs\piper\en_US-amy-medium.onnx"
OUTPUT_FILE = "output.wav"

def text_to_speech_piper(text):
    process = subprocess.run(
        [
            PIPER_PATH,
            "--model", MODEL_PATH,
            "--output_file", OUTPUT_FILE,
            "--length_scale", "1.2"
        ],
        input=text,
        text=True
    )

    os.system(f"start {OUTPUT_FILE}")

## Converting Answer into Speech with gTTs

In [55]:
from gtts import gTTS
import os

def text_to_speech_gtts(text):
    tts = gTTS(text)
    tts.save("response.mp3")
    os.system("start response.mp3")  

## Using Speech_text and Text_speech functions

In [58]:
record_audio()

text = speech_to_text()
print("Human:", text)


answer = ask_ollama(prompt=text)
print("AI Answer:", answer)

text_to_speech_piper(answer)

Speak now...
Human:  Medical assistant conversation
AI Answer: Here's a sample conversation for a medical assistant:

**Patient:** Hi, I'm here for my annual physical. I've been feeling a bit fatigued lately and wanted to get checked out.

**Medical Assistant (MA):** Okay! Let me just get your chart real quick. (checks the patient's chart) Alright, everything looks good so far. Can you tell me more about what you mean by fatigue? Is it constant or does it come and go?

**Patient:** It's been pretty consistent. I've been feeling like I need to take a nap every day around 2-3 pm.

**MA:** Okay, that doesn't sound normal at all! Have you noticed any other symptoms like pain or difficulty sleeping?

**Patient:** Actually, yeah... sometimes my joints ache and I have trouble falling asleep at night. And my hair has been falling out in clumps lately.

**MA:** Whoa, okay! That does sound concerning. Let me go get the doctor to come take a look. (gets the doctor)

**Doctor:** Hi there! So, what

## Build Memory

In [164]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

## disease Confidence

In [165]:
from collections import defaultdict
import numpy as np

def disease_confidence(ids, scores, all_chunks):
    disease_scores = defaultdict(list)

    for i, idx in enumerate(ids[0]):
        disease = all_chunks[idx]["metadata"]["disease"]
        disease_scores[disease].append(scores[0][i])
    final_scores = {
        disease: np.mean(vals)
        for disease, vals in disease_scores.items()
    }

    return sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

## NER for retrived Data

In [166]:
from transformers import pipeline
def NER_extract(text, ner):
    entites = ner(text)
    return entites

## Filter Entities

In [167]:
def filter_entities(entities, label):
    return [
        e["word"] for e in entities
        if e["entity_group"] == label and e["score"] > 0.8
    ]

## Retrieve Context

In [168]:
def retrieve_context(query, index, model, k=3):
    scores, ids = search(query, index, model, k)

    texts = []
    metas = []

    for i in ids[0]:
        texts.append(all_chunks[i]["text"])
        metas.append(all_chunks[i]["metadata"])

    return texts, metas, scores, ids

## Process Context

In [169]:
def process_context(texts, ner, query):
    context = "\n\n".join(texts)

    context_entities = NER_extract(context, ner)
    query_entities = NER_extract(query, ner)

    return {
        "context": context,
        "context_symptoms": list(set(filter_entities(context_entities, "Sign_symptom"))),
        "query_symptoms": list(set(filter_entities(query_entities, "Sign_symptom"))),
        "context_diseases": list(set(filter_entities(context_entities, "Disease"))),
        "query_diseases": list(set(filter_entities(query_entities, "Disease"))),
    }

## Build a prompt

In [183]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


def build_prompt():

    prompt = ChatPromptTemplate.from_messages([
        ("system",
        """
You are MedAssistant, a medical AI assistant.

IMPORTANT RULES:

INTENT:
- First detect the intent: (definition / diagnosis / symptoms / cause / advice)

REASONING:
- Use ONLY the provided context and extracted entities
- Prioritize extracted symptoms over raw text
- Do NOT assume or add missing symptoms
- If symptoms are insufficient → say uncertainty
- Prefer diseases supported by multiple symptoms
- Each disease must include a confidence percentage
- Explain reasoning briefly using matched symptoms

DIAGNOSIS:
- Do NOT return symptoms as diagnosis
- Return top 2–3 possible diseases
- Rank by likelihood using provided confidence

OUTPUT:
- Be concise (2–3 sentences max)
- Use natural spoken language (no bullet points or symbols)
- Keep explanation short and clear
- If multiple diseases share similar symptoms, mention this uncertainty

SAFETY:
- Do NOT provide definitive diagnosis
- Do NOT suggest medications
- If serious condition suspected → advise seeking medical help

FALLBACK:
- If answer not in context → say:
"The answer is not clearly available in the context."
        """
         ),
          MessagesPlaceholder(variable_name="history"),
        ("human",
         """

CONTEXT:
{context}

Symptoms (context):
{context_symptoms}

Diseases (context):
{context_diseases}

User symptoms:
{query_symptoms}

User diseases:
{query_diseases}

Disease likelihood:
{disease_conf}

QUESTION:
{query}

Answer clearly and concisely.
""")
    ])

    return prompt

## Test the whole Proccess

In [187]:
from langchain_ollama import ChatOllama

query = "what is cholera hi how are you doing today my name is ahmed and i want to know about cholera please tell me about cholera and its symptoms and causes and how can i treat it"

context_texts, context_metas, scores, ids = retrieve_context(query, index, model, k=3)

ner = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple")

context_data = process_context(context_texts, ner, query)
disease_conf = disease_confidence(ids, scores, all_chunks)


prompt = build_prompt()

In [ ]:
disease_conf[0]

NameError: name 'disease_conf' is not defined

: 

In [175]:
llm = ChatOllama(
    model="llama3:8b",
    temperature=0
)

chain = prompt | llm

In [176]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="history"
)

In [182]:
response = chain_with_memory.invoke(

    {
        "query": query,
        "context": context_data["context"],
        "context_symptoms": context_data["context_symptoms"],
        "context_diseases": context_data["context_diseases"],
        "query_symptoms": context_data["query_symptoms"],
        "query_diseases": context_data["query_diseases"],
        "disease_conf": disease_conf,
    },

    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response.content)

Your name is Abdelrahman.


In [178]:
print(response)

content="Hello Abdelrahman! As MedAssistant, I'm here to help you with any medical concerns or questions you may have.\n\nBased on the context provided, it seems like we're discussing a research study conducted at icddr,b in Dhaka during March-April 2002. The study focused on isolating Vibrio cholerae from stool samples.\n\nIf you'd like to discuss your symptoms or any health concerns you might be experiencing, please feel free to share them with me. I'll do my best to provide guidance and support.\n\nRemember, as a medical AI assistant, everything discussed remains confidential and for informational purposes only." additional_kwargs={} response_metadata={'model': 'llama3:8b', 'created_at': '2026-05-28T00:35:32.7143901Z', 'done': True, 'done_reason': 'stop', 'total_duration': 30898824800, 'load_duration': 121880500, 'prompt_eval_count': 203, 'prompt_eval_duration': 1011887500, 'eval_count': 128, 'eval_duration': 29640047500, 'logprobs': None, 'model_name': 'llama3:8b', 'model_provider'

## Get latitude and longitude

In [71]:
from winsdk.windows.devices.geolocation import Geolocator

async def get_coords():
    locator = Geolocator()
    pos = await locator.get_geoposition_async()
    return pos.coordinate.point.position.latitude, pos.coordinate.point.position.longitude

In [72]:
lat, lng = await get_coords()
print(f"latitude: {lat}, longitude: {lng}")

latitude: 30.466870789335562, longitude: 30.925501498349064


## Get closest hospital or clinic

In [73]:
def get_closest_facilities_safe(lat, lon, radius=20000):
    overpass_url = "https://overpass-api.de/api/interpreter"
    
    headers = {
        'User-Agent': 'Medical_Assistant',
        'Referer': 'https://overpass-turbo.eu/'
    }

    query = f"""
    [out:json][timeout:25];
    (
      node["amenity"~"hospital|clinic"](around:{radius}, {lat}, {lon});
      way["amenity"~"hospital|clinic"](around:{radius}, {lat}, {lon});
    );
    out center;
    """
    
    try:
        response = requests.get(overpass_url, params={'data': query}, headers=headers, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            return data 
        else:
            return f"{response.status_code}: {response.text}"
            
    except Exception as e:
        return e

In [74]:
closest_facilities = get_closest_facilities_safe(lat, lng, radius=10000)
closest_facilities

{'version': 0.6,
 'generator': 'Overpass API 0.7.62.11 87bfad18',
 'osm3s': {'timestamp_osm_base': '2026-05-26T12:24:00Z',
  'copyright': 'The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.'},
 'elements': [{'type': 'node',
   'id': 12043578527,
   'lat': 30.4679512,
   'lon': 30.9299433,
   'tags': {'addr:housenumber': '2',
    'addr:street': 'شارع الزراعه',
    'amenity': 'clinic',
    'name': 'Dr Diaa Nour Dental Clinic'}}]}

## format data and calculate distance 

In [75]:
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Geod

def process_overpass_results(data, user_lat, user_lon):
    elements = data.get('elements', [])

    features = []
    geod = Geod(ellps="WGS84")

    for elem in elements:
        if elem['type'] == 'node':
            e_lon, e_lat = elem['lon'], elem['lat']
        elif 'center' in elem:
            e_lon, e_lat = elem['center']['lon'], elem['center']['lat']
        else:
            continue
            
        _, _, distance = geod.inv(user_lon, user_lat, e_lon, e_lat)
        
        tags = elem.get('tags', {})
        name = tags.get('name', tags.get('name:ar', 'Unnamed Facility'))
        hospital_type = tags.get('amenity', 'hospital')
        
        features.append({
            'Name': name, 
            'Distance (km)': round(distance / 1000, 2),
            'Type': hospital_type,
        })

    gdf = gpd.GeoDataFrame(features)
    gdf = gdf.sort_values(by='Distance (km)').reset_index(drop=True)
    return gdf

In [76]:
final_hospitals = process_overpass_results(closest_facilities, lat, lng)
final_hospitals

,Name,Distance (km),Type
0,Dr Diaa Nour Dental Clinic,0.44,clinic


## Web Search

In [ ]:
from langchain_community.utilities import SerpAPIWrapper
google_search = SerpAPIWrapper(serpapi_api_key="f80974fb65334f54c4f50b1f7ca6bd5b17d90cdaf2a93c7094ffeb8bc96adcd8")

def web_search(query: str):
    """Search inside a specific website."""
    trusted_query = f"""
    site:who.int OR
    site:cdc.gov OR
    site:nih.gov OR
    site:mayoclinic.org OR
    site:pubmed.ncbi.nlm.nih.gov
    {query}
    """
    return google_search.run(trusted_query)

In [201]:
print(web_search("What is fever?"))

["A fever is a temporary rise in body temperature. It's one part of an overall response from the body's immune system. A fever is usually caused by an infection.", 'In physiological terms, fever has been defined as “a state of elevated core temperature, which is often, but not necessarily, part of the defensive response of ...', "Fever, or pyrexia, is the elevation of an individual's core body temperature above a 'set-point' regulated by the body's thermoregulatory center in the ...", "A fever is a rise in body temperature. It's often a sign of infection. Fever itself most often is harmless and it may play a role in ...", 'A fever (has a measured temperature of 100.4 °F [38 °C] or greater, or feels warm to the touch, or gives a history of feeling feverish) ...', 'Fever, an elevation above normal body temperature, is a frequent symptom of many infections [1]. It results from the release of endogenous pyrogens such as ...', 'A raised temperature is considered to be a fever at 38.5°C (101

In [1]:
query = "What is Cholera due to Vibrio cholerae 01, biovar cholerae?"

In [10]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

labels = [
    "medical diagnosis",
    "disease definition",
    "hospital search",
    "emergency",
    "general advice",
    "chatting"
]

query = "I have diarrhea and vomiting"

result = classifier(query, labels)

intent = result["labels"][0]

print(intent)
print(result)

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


disease definition
{'sequence': 'I have diarrhea and vomiting', 'labels': ['disease definition', 'emergency', 'medical diagnosis', 'chatting', 'hospital search', 'general advice'], 'scores': [0.45056766271591187, 0.2224290817975998, 0.19331596791744232, 0.059128887951374054, 0.04305300489068031, 0.03150537982583046]}


In [25]:
result = classifier("what is the nearest clinic?", labels)

intent = result["labels"][0]

In [26]:
intent

'medical diagnosis'

In [32]:
from langchain_community.utilities import SerpAPIWrapper
google_search = SerpAPIWrapper(serpapi_api_key="f80974fb65334f54c4f50b1f7ca6bd5b17d90cdaf2a93c7094ffeb8bc96adcd8")

def drug_search(query: str):
    """Search for drug information."""
    trusted_query = f"""
    site:mayoclinic.org OR
    site:nlm.nih.gov OR
    site:nhs.uk/medicines OR
    site:open.fda.gov/apis/drug OR
    site:dailymed.nlm.nih.gov
    {query}
    """
    return google_search.run(trusted_query)

In [30]:
drugs = drug_search("medicine for fever")

In [ ]:
import sounddevice as sd
import numpy as np
import queue

SAMPLE_RATE = 16000
BLOCK_SIZE = 1024

audio_queue = queue.Queue()


def callback(indata, frames, time, status):
    audio_queue.put(indata.copy())


def record_until_silence():
    print("🎤 Listening...")

    silence_threshold = 0.01
    silence_frames = 0
    max_silence = 60  

    audio_buffer = []
    speaking_started = False

    with sd.InputStream(samplerate=SAMPLE_RATE,
                        channels=1,
                        callback=callback,
                        blocksize=BLOCK_SIZE):

        while True:
            audio = audio_queue.get()
            audio_float = audio[:, 0]

            volume = np.abs(audio_float).mean()

            if volume > silence_threshold:
                speaking_started = True
                silence_frames = 0
                audio_buffer.append(audio_float)

            else:
                if speaking_started:
                    silence_frames += 1
                    audio_buffer.append(audio_float)

            if speaking_started and silence_frames > max_silence:
                break

    print("🛑 Stopped")

    return np.concatenate(audio_buffer)

In [4]:
import whisper

model = whisper.load_model("medium")

def transcribe(audio):
    result = model.transcribe(audio, fp16=False)
    return result["text"]

In [9]:
record = record_until_silence()

🎤 Listening...
🛑 Stopped


In [10]:
record

array([-0.00061035, -0.00021362, -0.0005188 , ...,  0.00073242,
        0.00256348,  0.003479  ], dtype=float32)

In [11]:
result = model.transcribe(record, fp16=False)

text = result["text"]

In [12]:
text

" Hi, how are you today? I'm talking to you about cholera. Cholera is a bacteria that causes dehydration and causes many symptoms also. So today we are going to discuss this disease."

In [ ]:
import sounddevice as sd
import numpy as np
import whisper

# ================= CONFIG =================
SAMPLE_RATE = 16000
BLOCK_SIZE = 1024

SILENCE_THRESHOLD = 0.01   # حساسية الصوت
MAX_SILENCE_FRAMES = 30    # ~1 ثانية سكوت

# ================= LOAD WHISPER =================
model = whisper.load_model("base")


# ================= RECORD FUNCTION =================
def record_until_silence():
    print("🎤 Listening... speak now")

    audio_buffer = []
    silence_counter = 0
    speaking_started = False

    def callback(indata, frames, time, status):
        nonlocal audio_buffer, silence_counter, speaking_started

        audio = indata[:, 0]

        volume = np.abs(audio).mean()

        # detect speech
        if volume > SILENCE_THRESHOLD:
            speaking_started = True
            silence_counter = 0
            audio_buffer.append(audio.copy())

        else:
            if speaking_started:
                silence_counter += 1
                audio_buffer.append(audio.copy())

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        callback=callback,
        blocksize=BLOCK_SIZE
    ):
        while True:
            sd.sleep(100)

            # stop condition
            if speaking_started and silence_counter > MAX_SILENCE_FRAMES:
                break

    print("🛑 Recording stopped")

    if len(audio_buffer) == 0:
        return None

    return np.concatenate(audio_buffer)


# ================= TRANSCRIBE =================
def speech_to_text():
    audio = record_until_silence()

    if audio is None:
        return ""

    print("🧠 Transcribing...")

    result = model.transcribe(audio, fp16=False)

    text = result["text"]

    print("📝 You said:", text)

    return text


# ================= RUN =================
if __name__ == "__main__":
    while True:
        query = speech_to_text()

        if query.strip():
            print("➡️ FINAL TEXT:", query)